In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [31]:
model = ChatGoogleGenerativeAI(model="models/gemini-3.1-flash-lite-preview")

In [37]:
#Define state
class BlogState(TypedDict):
    topic : str
    outline: str
    content: str
    evaluate: int

In [16]:
def gen_outline(state: BlogState) -> BlogState:

    #fetch the title
    title = state['topic']

    #write the prompt
    prompt = f"Generate a detailed outline on the given topic: {title}"

    #invoke the model
    response = model.invoke(prompt)

    outline = response.content[0]['text']

    #update the state
    state['outline'] = outline

    return state

In [36]:
def gen_blog(state: BlogState) -> BlogState:

    #fetch the title
    title = state['topic']

    #fetch the outline
    outline = state['outline']

    #write the prompt
    prompt = f"create a short and sweet blog on the topic: {title}, and its outline is: {outline}"

    #invoke the mode
    response = model.invoke(prompt)

    answer = response.content[0]['text']

    #update the state
    state['content'] = answer

    return state

In [38]:
def eval_blog(state: BlogState)->BlogState:

    #fetch the outline, content
    outline = state['outline']
    content = state['content']

    #prompt it
    prompt = f"based on my outline: {outline}, rate my blog content: {content}"

    #invoke the model
    response = model.invoke(prompt)

    answer = response.content[0]['text']

    state['evaluate'] = answer

    return state

In [39]:
#define graph nodes, edges and compile
graph = StateGraph(BlogState)

#define nodes
graph.add_node("gen_outline", gen_outline)
graph.add_node("gen_blog", gen_blog)
graph.add_node("eval_blog", eval_blog)

#define edges
graph.add_edge(START, "gen_outline")
graph.add_edge("gen_outline", "gen_blog" )
graph.add_edge("gen_blog", "eval_blog")
graph.add_edge("eval_blog", END)

workflow = graph.compile()
# graph.compile()

In [ ]:
initial_state = {'topic': "AI In India"}

final_state = workflow.invoke(initial_state)

print(final_state)





{'topic': 'AI In India', 'outline': 'This outline provides a comprehensive overview of the AI landscape in India, covering its current state, strategic direction, key sectors, challenges, and future trajectory.\n\n---\n\n# Outline: The Evolution and Impact of AI in India\n\n## I. Introduction\n*   **The AI Revolution:** Defining the global shift toward Artificial Intelligence.\n*   **India’s Competitive Edge:** Demographics (young workforce), digital infrastructure (India Stack), and a robust IT service backbone.\n*   **The "AI for All" Vision:** India’s national strategy to leverage AI for social inclusion and economic growth.\n\n## II. The Strategic Landscape\n*   **Government Initiatives:**\n    *   **IndiaAI Mission:** Overview of the $1.25 billion (₹10,372 crore) investment.\n    *   **MeitY’s Role:** Policy frameworks, ethical guidelines, and focus on sovereign AI.\n    *   **NITI Aayog’s Vision:** "AI for All" strategy focusing on healthcare, agriculture, education, and infrastr

In [41]:
print(final_state['evaluate'])

Your blog post is **excellent**. You have successfully distilled a complex, multifaceted outline into a punchy, readable, and professional narrative. It hits the perfect balance between "big-picture vision" and "grounded reality."

Here is my rating and a few suggestions to make it even better:

### **Rating: 9/10**

**What you did well:**
*   **Narrative Flow:** You established a clear arc—starting with the infrastructure foundation, moving to real-world impact, acknowledging the friction, and ending with a strong, visionary closing statement.
*   **Tone:** The tone is professional, optimistic, and authoritative. It feels like an executive summary written for a tech-savvy audience.
*   **The "India Stack" hook:** Linking the success of UPI/Aadhaar to the potential of AI is your strongest argument. It gives the reader a concrete mental model for why India is uniquely positioned for this.
*   **Conciseness:** You avoided the trap of getting bogged down in too much jargon, keeping the fo